Import the Data

In [287]:
import pandas as pd
import numpy as np
import os
import textwrap
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer

In [288]:
data_dir = 'data_readinglevel'
x_train_df = pd.read_csv(os.path.join(data_dir, 'x_train.csv'))
y_train_df = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))


# take the first 969 rows to use as validation
x_valid_train = x_train_df.head(968).copy(deep=True)

# take the rest for training
x_train_new = x_train_df.iloc[968:].copy()
x_train_new = x_train_new.reset_index(drop=True)


x_test_df = pd.read_csv(os.path.join(data_dir, 'x_test.csv'))

N, n_cols = x_train_df.shape
print("Shape of x_train_df: (%d, %d)" % (N, n_cols))
print("Shape of y_train_df: %s" % str(y_train_df.shape))

Shape of x_train_df: (5557, 32)
Shape of y_train_df: (5557, 5)


In [ ]:
x_valid_train

In [ ]:
x_train_new

Add the labels to Y

In [291]:
# Create the binary classification column y
# 2-3 is labeled as 0
# 4-5 is labeled as 1

y_train_df["true_labels"] = np.where(y_train_df["Coarse Label"].str.contains("Key Stage 2-3"), 0, 1)
y = y_train_df.true_labels.to_numpy() 

y_new = y[968:].copy()
y_test = y[:968].copy()

Function to split the text and numerical columns we want to use as features

In [292]:
def make_preprocessor(text_cols, numeric_cols):
    transformers = []
    
    if text_cols:
        transformers.append(('text', CountVectorizer(ngram_range=(1,1), lowercase= True), text_cols))
    
    if numeric_cols:
        transformers.append(('num', MinMaxScaler(), numeric_cols))
    
    return ColumnTransformer(transformers=transformers)

Pick the Features we want to do and make the exact preprocessing

In [293]:
feature_columns = [
    "avg_word_length",
    "avg_sentence_length",
    "info_type_token_ratio",
    "readability_FleschReadingEase",
    "readability_DaleChallIndex", "punctuation_frequency"
]

text_columns = None

preprocessor = make_preprocessor(text_columns, feature_columns)

Define the pipeline

In [294]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])


Define the KFold

In [295]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

Define Grid Search

In [296]:
# With the Text
#param_grid = {
#    'classifier__C': [0.01, 0.1, 1, 10, 100],
#    'preprocessor__text__min_df': [10,11,12],       # ignore words appearing in <min_df docs
#    'preprocessor__text__max_df': [0.8, 1.0]    # ignore words appearing in > max_df fraction
#}

# Without the text
param_grid = {
    'classifier__C': np.logspace(-4, 4, 9)
}
grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

Fit the Model

In [297]:
grid.fit(x_train_new, y_new)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.",{'classifier__C': array([1.e-04... 1.e+04])}
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is als

Results: 

In [298]:
print("Best Parameters:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

Best Parameters: {'classifier__C': np.float64(1000.0)}
Best CV ROC-AUC: 0.7139458807554129


In [299]:
# Get the fitted pipeline
best_pipeline = grid.best_estimator_
classifier = best_pipeline.named_steps['classifier']
preprocessor = best_pipeline.named_steps['preprocessor']

# Text features
if 'text' in preprocessor.named_transformers_:
    vectorizer = preprocessor.named_transformers_['text']
    text_features = vectorizer.get_feature_names_out()
    text_coefs = classifier.coef_[0][:len(text_features)]
else:
    text_features = []
    text_coefs = []

# Numeric features
if 'num' in preprocessor.named_transformers_:
    num_features = feature_columns
    num_coefs = classifier.coef_[0][-len(num_features):]
else:
    num_features = []
    num_coefs = []

# Combine into a DataFrame
all_features = list(text_features) + list(num_features)
all_coefs = list(text_coefs) + list(num_coefs)

feature_importance = pd.DataFrame({
    'feature': all_features,
    'coefficient': all_coefs
})

# Sort by absolute importance
feature_importance['abs_coef'] = feature_importance['coefficient'].abs()
feature_importance = feature_importance.sort_values('abs_coef', ascending=False).drop(columns='abs_coef')

print("\nMost important features (text + numeric):")
print(feature_importance)

numeric_importance = feature_importance[feature_importance['feature'].isin(feature_columns)]
print("\nMost important numeric features:")
print(numeric_importance)


Most important features (text + numeric):
                         feature  coefficient
1            avg_sentence_length    12.280868
3  readability_FleschReadingEase    10.056661
4     readability_DaleChallIndex     5.266340
5          punctuation_frequency    -3.239844
0                avg_word_length     0.954142
2          info_type_token_ratio    -0.100954

Most important numeric features:
                         feature  coefficient
1            avg_sentence_length    12.280868
3  readability_FleschReadingEase    10.056661
4     readability_DaleChallIndex     5.266340
5          punctuation_frequency    -3.239844
0                avg_word_length     0.954142
2          info_type_token_ratio    -0.100954


In [300]:
y_proba1_test = grid.best_estimator_.predict_proba(x_test_df)[:, 1]
print(y_proba1_test)
np.savetxt('yproba1_test.txt', y_proba1_test)

[0.63569001 0.64189864 0.59492802 ... 0.33492291 0.32275441 0.48155473]


In [301]:
grid.score(x_valid_train, y_test)

0.8206821898247312